## Smiles to SDF

here we just create SDFs from Andreas SMILES

In [30]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import SDWriter


In [12]:

metadata_file = r"../../data/raw/docking/Benchmark_Bionemo_CS_targets_list(Sheet1).csv"

df_meta = pd.read_csv(metadata_file)
df_meta.head(5)

,Indication,Organism,Protein Superfamily/family,Name of target,Cofactors,Ligand(s),Canonical SMILES Ligand,Chains,Comments,Internal pdb code,Uniprot sequence,Resolution [A],Schrodinger prot prep - Glide docking,Flare prot prep - Flare docking,Schrodinger prot prep - Flare docking,Flare prot prep - Glide docking
0,H,Chlamydomonas reinhardtii,solanesyl diphosphate\nsynthase,SPS,NaN,apo,NaN,Monomer,NaN,bcsin_110.pdb,A8HRQ4,1.7,N.A.,N.A.,N.A.,N.A.
1,H,Arabidopsis thaliana,phosphatase,pyr1-hab1,NaN,BCS-CT37643,CCSP(=S)(c1ccccc1)C2(CC2)C#N,Heterodimer,"Many structures in pdb, but not for this chemi...",bcsin_226.pdb,"O49686, Q9CAJ0",2.3,OK (8 poses generated),OK (10 poses generated),OK (10 poses generated),NaN
2,H,Arabidopsis thaliana,phosphatase,pyr1-hab1,NaN,BCS-CN76566,Cc1ccccc1CNC(=O)C2CCc3nnc(c4ccc(Cl)cc4)n23,Heterodimer,"Many structures in pdb, but not for this chemi...",bcsin_235.pdb,"O49686, Q9CAJ0",2.8,OK (6 poses generated),"OK (8 poses T1, 8 poses T2)","OK (9 poses T1, 9 poses T2)",NaN
3,H,Arabidopsis thaliana,dioxygenase,HPPD,FE2,BCS-CV49374,Cc1c2SCCC(=O)c2ccc1C(=O)Nc3nnnn3C,Homodimer,Fe2 required for ligand binding,bcsin_233.pdb,P93836,1.9,OK (7 poses generated),OK (7 poses generated),OK (7 poses generated),NaN
4,H,Arabidopsis thaliana,tRNA synthetase,KRS1,LYS,BCS-CY86357,Nc1nsc(\C(=N\C#N)\NCC2CCCCC2)c1Cl,Homodimer,NaN,bcsin_261.pdb,Q9ZPI1,1.9,OK (10 poses generated),"OK (7 poses T1, 9 poses T2, 10 poses T3, 8 pos...","OK (5 poses T1, 5 poses T2, 7 poses T3, 8 pose...",NaN


In [13]:
df_meta.shape

(15, 16)

In [17]:
df_data = df_meta[["Ligand(s)","Canonical SMILES Ligand"]].copy()
print(f" shape before droping NaNs: {df_data.shape}")
df_data.dropna(inplace=True)
print(f" shape AFTER droping NaNs: {df_data.shape}")

 shape before droping NaNs: (15, 2)
 shape AFTER droping NaNs: (13, 2)


In [24]:
df_data.head()

,Ligand(s),Canonical SMILES Ligand
1,BCS-CT37643,CCSP(=S)(c1ccccc1)C2(CC2)C#N
2,BCS-CN76566,Cc1ccccc1CNC(=O)C2CCc3nnc(c4ccc(Cl)cc4)n23
3,BCS-CV49374,Cc1c2SCCC(=O)c2ccc1C(=O)Nc3nnnn3C
4,BCS-CY86357,Nc1nsc(\C(=N\C#N)\NCC2CCCCC2)c1Cl
5,BCS-CZ81034,O=C(NCC1CCCCC1)c2snc3ncccc23


In [25]:
# crosscheck Read smiles from Excel and drop NaNs 
smiles = df_data["Canonical SMILES Ligand"].unique()
bayer_ids = smiles = df_data["Ligand(s)"].unique()
print(len(smiles))
print(len(bayer_ids))

13
13


## 2D Generation

In [33]:
# Directory to save SDF files
output_dir = '../../data/raw/docking/ligands'
os.makedirs(output_dir, exist_ok=True)

In [34]:


# Create SDF files, one for each unique ID
for index, row in df_data.iterrows():
    smiles = row['Canonical SMILES Ligand']
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is not None:
        sdf_file_path = os.path.join(output_dir, f"{row['Ligand(s)']}.sdf")
        with Chem.SDWriter(sdf_file_path) as writer:
            # Optionally, you can add properties to the molecule
            mol.SetProp('ID', str(row['Ligand(s)']))
            writer.write(mol)
    else:
        print(f"Cant create mol form sequenc with id: {row['Ligand(s)']}")

print(f"SDF files created in directory: {output_dir}")


SDF files created in directory: ../../data/raw/docking/ligands


## 3D SDF Generation

In [35]:
# Directory to save SDF files
output_dir2 = '../../data/raw/docking/ligands_3d'
os.makedirs(output_dir2, exist_ok=True)

In [36]:
# Create SDF files, one for each unique ID
for index, row in df_data.iterrows():
    smiles = row['Canonical SMILES Ligand']
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is not None:
        # Generate 3D coordinates
        mol = Chem.AddHs(mol)  # Add hydrogens
        AllChem.EmbedMolecule(mol)  # Generate 3D conformation
        AllChem.UFFOptimizeMolecule(mol)  # Optional: Optimize the 3D structure

        sdf_file_path = os.path.join(output_dir2, f"{row['Ligand(s)']}.sdf")
        with Chem.SDWriter(sdf_file_path) as writer:
            # Optionally, you can add properties to the molecule
            mol.SetProp('ID', str(row['Ligand(s)']))
            writer.write(mol)
    else:
        print(f"Cant create mol form sequenc with id: {row['Ligand(s)']}")
    

print(f"SDF files created in directory: {output_dir2}")

SDF files created in directory: ../../data/raw/docking/ligands_3d
